## Phase 1: Data Harmonization & Stratified Splitting

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

In [2]:
# --- CONFIGURATION ---
TRAIN_FILE = "google_drive/train.csv"   # Replace with actual path
TEST_FILE = "google_drive/test.csv"     # Replace with actual path
OUTPUT_FILE = "unified_dta_competition_split.csv"

In [3]:
def parse_split_membership(df, col_name):
    """Parses 'A, B, C' string formats into explicit binary columns."""
    print("Parsing split memberships...")
    # Remove whitespace
    clean_splits = df[col_name].astype(str).str.replace(' ', '')
    
    df['in_A'] = clean_splits.apply(lambda x: 1 if 'A' in x.split(',') else 0)
    df['in_B'] = clean_splits.apply(lambda x: 1 if 'B' in x.split(',') else 0)
    df['in_C'] = clean_splits.apply(lambda x: 1 if 'C' in x.split(',') else 0)
    return df

In [4]:
print("Loading datasets...")
df_train = pd.read_csv(TRAIN_FILE)
df_test = pd.read_csv(TEST_FILE)

Loading datasets...


In [6]:
# Standardize dfs to match
df_train = df_train.rename(columns={'sample_id': 'id'})
print(df_train.head())
df_test = df_test.rename(columns={'split': 'split_membership'})
df_test['affinity'] = np.nan # Test set lacks affinity labels
print(df_test.head())

        id                                             smiles  \
0  train_0               CN1CCN(C(=O)c2cc3cc(Cl)ccc3[nH]2)CC1   
1  train_1  CS(=O)c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc3)[nH...   
2  train_2          Cn1cc(-c2ccc3nnc(Sc4ccc5ncccc5c4)n3n2)cn1   
3  train_3   Cc1cc2c(F)c(Oc3ncnn4cc(OCC(C)O)c(C)c34)ccc2[nH]1   
4  train_4  Cc1cc(Nc2cc(N3CCN(C)CC3)nc(Sc3ccc(NC(=O)C4CC4)...   

                                         protein_seq  affinity  \
0  MVDMGALDNLIANTAYLQARKPSDCDSKELQRRRRSLALPGLQGCA...  6.086186   
1  MESPASSQPASMPQSKGKSKRKKDLRISCMSKPPAPNPTPPRNLDS...  5.000000   
2  MSVLGEYERHCDSINSDFGSESGGCGDSSPGPSASQGPRAGGGAAE...  5.000000   
3  MKNIYCLIPKLVNFATLGCLWISVVQCTVLNSCLKSCVTNLGQQLD...  5.000000   
4  MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFER...  5.356547   

  split_membership  
0            A,B,C  
1              A,C  
2              A,C  
3            A,B,C  
4            A,B,C  
           id                                             smiles  \
0  D_99418882  CN(

In [7]:
# Parse memberships
df_train = parse_split_membership(df_train, 'split_membership')
print(df_train.head())
df_test = parse_split_membership(df_test, 'split_membership')
print(df_test.head())

Parsing split memberships...
        id                                             smiles  \
0  train_0               CN1CCN(C(=O)c2cc3cc(Cl)ccc3[nH]2)CC1   
1  train_1  CS(=O)c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc3)[nH...   
2  train_2          Cn1cc(-c2ccc3nnc(Sc4ccc5ncccc5c4)n3n2)cn1   
3  train_3   Cc1cc2c(F)c(Oc3ncnn4cc(OCC(C)O)c(C)c34)ccc2[nH]1   
4  train_4  Cc1cc(Nc2cc(N3CCN(C)CC3)nc(Sc3ccc(NC(=O)C4CC4)...   

                                         protein_seq  affinity  \
0  MVDMGALDNLIANTAYLQARKPSDCDSKELQRRRRSLALPGLQGCA...  6.086186   
1  MESPASSQPASMPQSKGKSKRKKDLRISCMSKPPAPNPTPPRNLDS...  5.000000   
2  MSVLGEYERHCDSINSDFGSESGGCGDSSPGPSASQGPRAGGGAAE...  5.000000   
3  MKNIYCLIPKLVNFATLGCLWISVVQCTVLNSCLKSCVTNLGQQLD...  5.000000   
4  MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFER...  5.356547   

  split_membership  in_A  in_B  in_C  
0            A,B,C     1     1     1  
1              A,C     1     0     1  
2              A,C     1     0     1  
3            A,B,C     1   

In [8]:
# Stratified split for internal validation
print("Creating stratified internal validation set...")
train_split, val_split = train_test_split(
    df_train,
    test_size=0.10,
    random_state=42,
    stratify=df_train['split_membership'] # Ensures exact representation of all A/B/C intersections
)

train_split['Data_Split'] = 'Train'
val_split['Data_Split'] = 'Validation'
df_test['Data_Split'] = 'Test'

Creating stratified internal validation set...


In [9]:
train_split

,id,smiles,protein_seq,affinity,split_membership,in_A,in_B,in_C,Data_Split
22215,train_22215,Nc1nc(N)c2nc(-c3cccc(O)c3)c(-c3cccc(O)c3)nc2n1,MSFLSRQQPPPPRRAGAACTLRQKLIFSPCSDCEEEEEEEEEEGSG...,5.000000,"B,C",0,1,1,Train
7731,train_7731,COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1,MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDRE...,5.000000,"A,B,C",1,1,1,Train
13281,train_13281,CCN(CCO)CCCOc1ccc2c(Nc3cc(CC(=O)Nc4cccc(F)c4)[...,MPPKRNEKYKLPIPFPEGKVLDDMEGNQWVLGKKIGSGGFGLIYLA...,5.000000,"A,B,C",1,1,1,Train
3574,train_3574,Cn1cc(C2=C(c3cn(C4CCN(Cc5ccccn5)CC4)c4ccccc34)...,MATCIGEKIEDFKVGNLLGKGSFAGVYRAESIHTGLEVAIKMIDKK...,5.000000,"A,B,C",1,1,1,Train
17545,train_17545,CNC1CC2OC(C)(C1OC)n1c3ccccc3c3c4c(c5c6ccccc6n2...,MAAAETQSLREQPEMEDANSEKSINEENGEVSEDQSQNKHSRHKKK...,6.657577,"A,B,C",1,1,1,Train
...,...,...,...,...,...,...,...,...,...
22678,train_22678,O=c1ncn2nc(Sc3ccc(F)cc3F)ccc2c1-c1c(Cl)cccc1Cl,MLTRKPSAAAPAAYPTGRGGDSAVRQLQASPGLGAGATRSGVGTGP...,5.000000,"B,C",0,1,1,Train
5295,train_5295,O=C(c1ccc(C=Cc2n[nH]c3ccccc23)cc1)N1CCNCC1,MILIPRMLLVLFLLLPILSSAKAQVNPAICRYPLGMSGGQIPDEDI...,5.677781,"A,B,C",1,1,1,Train
8844,train_8844,Nc1nc(N)c2nc(-c3cccc(O)c3)c(-c3cccc(O)c3)nc2n1,MTSSLQRPWRVPWLPWTILLVSTAAASQNQERLCAFKDPYQQDLGI...,5.000000,"A,B,C",1,1,1,Train
8118,train_8118,CCN1CCN(Cc2ccc(NC(=O)Nc3ccc(Oc4cc(NC)ncn4)cc3)...,MGFGSDLKNSHEAVLKLQDWELRLLETVKKFMALRIKSDKEYASTL...,6.229148,"A,B,C",1,1,1,Train


In [10]:
# Calculate Sampling Weights for Group DRO (Training set only)
# Inverse frequency weighting: (Total Samples) / (Number of Samples in this specific subset group)
print("Calculating batch sampling weights for minority subsets...")
subset_counts = train_split['split_membership'].value_counts()
total_train = len(train_split)
weight_map = {subset: total_train / count for subset, count in subset_counts.items()}

train_split['sample_weight'] = train_split['split_membership'].map(weight_map)
val_split['sample_weight'] = 1.0
df_test['sample_weight'] = 1.0

Calculating batch sampling weights for minority subsets...


In [13]:
subset_counts

split_membership
A,B,C    11249
A,C       4987
B,C       4782
C         2176
Name: count, dtype: int64

In [12]:
weight_map

{'A,B,C': 2.0618721664147923,
 'A,C': 4.650892320032083,
 'B,C': 4.850271852781263,
 'C': 10.659007352941176}

In [11]:
train_split

,id,smiles,protein_seq,affinity,split_membership,in_A,in_B,in_C,Data_Split,sample_weight
22215,train_22215,Nc1nc(N)c2nc(-c3cccc(O)c3)c(-c3cccc(O)c3)nc2n1,MSFLSRQQPPPPRRAGAACTLRQKLIFSPCSDCEEEEEEEEEEGSG...,5.000000,"B,C",0,1,1,Train,4.850272
7731,train_7731,COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1,MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDRE...,5.000000,"A,B,C",1,1,1,Train,2.061872
13281,train_13281,CCN(CCO)CCCOc1ccc2c(Nc3cc(CC(=O)Nc4cccc(F)c4)[...,MPPKRNEKYKLPIPFPEGKVLDDMEGNQWVLGKKIGSGGFGLIYLA...,5.000000,"A,B,C",1,1,1,Train,2.061872
3574,train_3574,Cn1cc(C2=C(c3cn(C4CCN(Cc5ccccn5)CC4)c4ccccc34)...,MATCIGEKIEDFKVGNLLGKGSFAGVYRAESIHTGLEVAIKMIDKK...,5.000000,"A,B,C",1,1,1,Train,2.061872
17545,train_17545,CNC1CC2OC(C)(C1OC)n1c3ccccc3c3c4c(c5c6ccccc6n2...,MAAAETQSLREQPEMEDANSEKSINEENGEVSEDQSQNKHSRHKKK...,6.657577,"A,B,C",1,1,1,Train,2.061872
...,...,...,...,...,...,...,...,...,...,...
22678,train_22678,O=c1ncn2nc(Sc3ccc(F)cc3F)ccc2c1-c1c(Cl)cccc1Cl,MLTRKPSAAAPAAYPTGRGGDSAVRQLQASPGLGAGATRSGVGTGP...,5.000000,"B,C",0,1,1,Train,4.850272
5295,train_5295,O=C(c1ccc(C=Cc2n[nH]c3ccccc23)cc1)N1CCNCC1,MILIPRMLLVLFLLLPILSSAKAQVNPAICRYPLGMSGGQIPDEDI...,5.677781,"A,B,C",1,1,1,Train,2.061872
8844,train_8844,Nc1nc(N)c2nc(-c3cccc(O)c3)c(-c3cccc(O)c3)nc2n1,MTSSLQRPWRVPWLPWTILLVSTAAASQNQERLCAFKDPYQQDLGI...,5.000000,"A,B,C",1,1,1,Train,2.061872
8118,train_8118,CCN1CCN(Cc2ccc(NC(=O)Nc3ccc(Oc4cc(NC)ncn4)cc3)...,MGFGSDLKNSHEAVLKLQDWELRLLETVKKFMALRIKSDKEYASTL...,6.229148,"A,B,C",1,1,1,Train,2.061872


In [14]:
# Merge and Save
df_unified = pd.concat([train_split, val_split, df_test], ignore_index=True)
df_unified.to_csv(OUTPUT_FILE, index=False)

print("\nFinal Data Split Distribution:")
print(df_unified['Data_Split'].value_counts())
print(f"\nPhase 1 complete. Dataset saved to {OUTPUT_FILE}")


Final Data Split Distribution:
Data_Split
Train         23194
Test          23191
Validation     2578
Name: count, dtype: int64

Phase 1 complete. Dataset saved to unified_dta_competition_split.csv


In [15]:
df_unified

,id,smiles,protein_seq,affinity,split_membership,in_A,in_B,in_C,Data_Split,sample_weight
0,train_22215,Nc1nc(N)c2nc(-c3cccc(O)c3)c(-c3cccc(O)c3)nc2n1,MSFLSRQQPPPPRRAGAACTLRQKLIFSPCSDCEEEEEEEEEEGSG...,5.000000,"B,C",0,1,1,Train,4.850272
1,train_7731,COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1,MQYLNIKEDCNAMAFCAKMRSSKKTEVNLEAPEPGVEVIFYLSDRE...,5.000000,"A,B,C",1,1,1,Train,2.061872
2,train_13281,CCN(CCO)CCCOc1ccc2c(Nc3cc(CC(=O)Nc4cccc(F)c4)[...,MPPKRNEKYKLPIPFPEGKVLDDMEGNQWVLGKKIGSGGFGLIYLA...,5.000000,"A,B,C",1,1,1,Train,2.061872
3,train_3574,Cn1cc(C2=C(c3cn(C4CCN(Cc5ccccn5)CC4)c4ccccc34)...,MATCIGEKIEDFKVGNLLGKGSFAGVYRAESIHTGLEVAIKMIDKK...,5.000000,"A,B,C",1,1,1,Train,2.061872
4,train_17545,CNC1CC2OC(C)(C1OC)n1c3ccccc3c3c4c(c5c6ccccc6n2...,MAAAETQSLREQPEMEDANSEKSINEENGEVSEDQSQNKHSRHKKK...,6.657577,"A,B,C",1,1,1,Train,2.061872
...,...,...,...,...,...,...,...,...,...,...
48958,D_10096613,Cn1cc(-c2ccc3nnc(Sc4ccc5ncccc5c4)n3n2)cn1,MLSNSQGQSPPVPFPAPAPPPQPPTPALPHPPAQPPPPPPQQFPQF...,NaN,B,0,1,0,Test,1.000000
48959,D_39322608,CC(C)(C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc(Cl)c4)...,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...,NaN,C,0,0,1,Test,1.000000
48960,D_58207541,Cc1cnc(Nc2ccc(OCCN3CCCC3)cc2)nc1Nc1cccc(S(=O)(...,MRGARGAWDFLCVLLLLLRVQTGSSQPSVSPGEPSPPSIHPGKSDL...,NaN,A,1,0,0,Test,1.000000
48961,D_91520297,CC(C)(C)c1cnc(CSc2cnc(NC(=O)C3CCNCC3)s2)o1,MAHLRGFANQHSRVDPEELFTKLDRIGKGSFGEVYKGIDNHTKEVV...,NaN,A,1,0,0,Test,1.000000
